# Notebook 04 — Skills

**A skill is a folder.** We'll prove that the folder form is cheaper, more
reliable, and shippable than the giant prompt you've been writing.

> **Where this sits in the five moves (sec 1):** a skill is **STORE +
> RETRIEVE + INSERT on demand** — procedural memory written down once (store),
> pulled in only when the task matches (retrieve), and placed in the window in
> the right form and order (insert). Progressive disclosure *is* retrieval
> applied to your own instructions. The packaging is Anthropic's "Agent
> Skills" pattern ([Anthropic, *Effective context engineering for AI agents*,
> 2025](https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents);
> LangChain's "Write / Select" maps onto store / retrieve,
> [Martin, 2025](https://www.langchain.com/blog/context-engineering-for-agents)).

The proof, in five moves:
1. Run the **same task** as a bloated system prompt vs a progressively-disclosed skill, and measure the token gap.
2. Walk a real `SKILL.md` folder.
3. Push one step down from LLM to script — and show it gets cheaper *and* more reliable.
4. Run the validator + tests live.
5. Scaffold a skill the way you actually build one.


## Setup

`setup()` connects the model and counts tokens. The rest of this cell just names the demo skill's folders and loads its scrap-log data. Run it once.

In [ ]:
from pathlib import Path
import json, re, subprocess, sys, importlib.util
from textwrap import dedent

from roadshow import setup
from roadshow_viz import bars, compare, stacked, matrix
from arcrun import run, Tool, ToolContext, StaticProvider

# model: the connected LLM · ask: one-shot helper · ntok: token counter · DATA: data/ path
model, ask, ntok, DATA = setup()

# Lesson paths: the demo skill we inspect/run, and its scrap-log data.
SKILLS_DIR = DATA.parent / "skills"
SKILL_DIR  = SKILLS_DIR / "scrap-coder"
SCRAP_DIR  = DATA / "scrap"

# Noisy real-world rows used for the deterministic-vs-LLM math demo (live, no fallbacks).
NOISY_INPUTS = json.loads((SCRAP_DIR / "noisy_inputs.json").read_text())

print("skill dir:", SKILL_DIR, "exists:", SKILL_DIR.exists())
print("noisy rows:", len(NOISY_INPUTS))

## ▶ Control cell — the only cell you edit

Every knob the notebook reads lives here. Change one, then **re-run this cell and the cells below it** to see the effect. The model itself is already wired by `setup()` above — this cell only steers *what* we run.

In [2]:
# ════════════════ CONTROL CELL ════════════════
APPROACH   = "skill"        # which form to run: "bloated_prompt" | "skill"
SKILL_NAME = "scrap-coder"  # the demo skill we inspect and run
PUSH_DOWN  = True           # push-down demo: compare script vs LLM on the math step
TASK       = "Categorize this shift's scrap log into the standard 7 codes and total cost."
# ═══════════════════════════════════════════════

## The two approaches

**`bloated_prompt`** — one large system prompt holding *all* 7 scrap codes, the
cost table, the output format, and the edge-case rules, **paid on every call.**

**`skill`** — a `scrap-coder/` folder: a tiny `SKILL.md` router +
`knowledge/scrap_codes.md` + `scripts/total_cost.py` + a template, loaded
progressively (only what the task needs).

The bloated prompt isn't just expensive. Over-stuffing the window is the
**confusion** failure mode from sec 1
([Breunig, *How Contexts Fail*, 2025](https://www.dbreunig.com/2025/06/22/how-contexts-fail-and-how-to-fix-them.html)):
more rules and tools in front of the model than the task needs drags response
quality down. We measure the token counts from the *actual* files below.


Build both forms from the **same** knowledge, then run the same task through each. The answer matters less than the cost and reliability we measure next.

**Read the shared knowledge from the skill folder.** One codes table, one `SKILL.md`, one output template — the raw material both approaches are built from.

In [ ]:
# The shared knowledge, read from the skill folder.
SCRAP_CODES_MD = (SKILL_DIR / "knowledge" / "scrap_codes.md").read_text()
SKILL_MD       = (SKILL_DIR / "SKILL.md").read_text()
TEMPLATE_MD    = (SKILL_DIR / "templates" / "shift_report.md").read_text()

# Split SKILL.md into its router (YAML front-matter) and its body (the procedure).
_fm = re.search(r"^---\n(.*?)\n---\n?", SKILL_MD, re.DOTALL)
SKILL_FRONTMATTER = _fm.group(1) if _fm else ""           # the router "card"
SKILL_BODY        = SKILL_MD[_fm.end():] if _fm else SKILL_MD  # Files/Contract/Steps/...

print("codes table:", ntok(SCRAP_CODES_MD), "tok | SKILL.md body:", ntok(SKILL_BODY), "tok")

**Build approach A — the bloated prompt.** One giant system block: the full code table, every procedural rule, *and* the output template, inlined and paid on **every** call. This is the realistic "few hundred lines" anti-pattern.

In [ ]:
# bloated_prompt: ONE giant block — everything inlined, paid on EVERY call.
BLOATED_SYSTEM = dedent("""\
    You are a scrap-coding assistant for Line 4 swing shift. Apply ALL of the
    following on EVERY request, in full, with no omissions:

    """) + SCRAP_CODES_MD + "\n\n" + SKILL_BODY + "\n\nOutput in this exact shape:\n" + TEMPLATE_MD

print("bloated system block:", ntok(BLOATED_SYSTEM), "tokens")

**Build approach B — the skill.** Progressive disclosure: the **router** (front-matter) loads up front so the model can *decide* to use the skill; because this task matches, it then pulls the **one** knowledge file it needs — and nothing else (no full procedure, no template).

In [ ]:
# skill: progressive disclosure — router up front, then ONLY the one matched file.
SKILL_ROUTER = dedent("""\
    You are a scrap-coding assistant. A skill matched this task. Its router card:

    """) + SKILL_FRONTMATTER

SKILL_SYSTEM = SKILL_ROUTER + dedent("""

    The skill matched, so its knowledge file is now loaded:

    """) + SCRAP_CODES_MD + dedent("""

    Code each row to exactly one of S1-S7, exclude rework, default qty to 1,
    and give a per-code breakdown and a dollar total using the unit costs above.
    """)

SAMPLE_LOG = "\n".join(NOISY_INPUTS[:6])   # a small shift slice for the live run
print("skill router:", ntok(SKILL_ROUTER), "tok | full skill system:", ntok(SKILL_SYSTEM), "tok")

**A helper to read the model's stated total.** Cheaper must not mean *wrong*: the skill has to reach the **same** dollar figure as the bloated prompt. This pulls the largest stated `$` from an answer so we can check that.

In [ ]:
# Pull the dollar total the model reported (its largest stated $ figure).
def stated_total(text: str):
    nums = [float(x.replace(",", "")) for x in re.findall(r"-?[\d,]+\.\d{2}", text or "")]
    return max(nums) if nums else None

**Give the loop one placeholder tool.** The agentic loop needs *a* capability to advertise; the lesson here is the prompt size, not the tool, so this `noop` just stands in.

In [ ]:
async def _noop(args: dict, ctx: ToolContext) -> str:
    """Placeholder tool so the agentic loop has a capability to advertise."""
    return "noop"

NOOP_TOOL = Tool(
    name="noop", description="placeholder so the loop has a tool",
    input_schema={"type": "object", "properties": {}},
    execute=_noop,
)

**One function to run the task either way.** It records the loaded token count, the answer, and the stated dollar total.

In [ ]:
async def run_task(task: str, approach: str) -> dict:
    """Run the task one way; record loaded tokens, the answer, and the stated total.

    loaded_tokens is the realistic in-context size for this matched task:
    bloated = everything; skill = router + the one knowledge file it pulled."""
    system = BLOATED_SYSTEM if approach == "bloated_prompt" else SKILL_SYSTEM
    rec = {"approach": approach, "loaded_tokens": ntok(system)}
    result = await run(
        model,
        StaticProvider([NOOP_TOOL]),
        system,
        f"{task}\n\nScrap log:\n{SAMPLE_LOG}",
        max_turns=4,
    )
    rec["content"]  = result.content
    rec["cost_usd"] = getattr(result, "cost_usd", None)
    rec["total"]    = stated_total(result.content)
    return rec

**Run both approaches** on the same task, then check the skill reached the *same* total as the bloated prompt — cheaper **and** right, not cheaper-but-wrong.

In [ ]:
RESULTS = {}
for approach in ["bloated_prompt", "skill"]:
    RESULTS[approach] = await run_task(TASK, approach=approach)

SKILL_AGREES = (RESULTS["bloated_prompt"]["total"] is not None
                and RESULTS["bloated_prompt"]["total"] == RESULTS["skill"]["total"])

print("bloated loaded: %d tokens  | total $%s"
      % (RESULTS["bloated_prompt"]["loaded_tokens"], RESULTS["bloated_prompt"]["total"]))
print("skill   loaded: %d tokens  | total $%s"
      % (RESULTS["skill"]["loaded_tokens"], RESULTS["skill"]["total"]))
print("skill matches bloated's total:", SKILL_AGREES, "(cheaper AND right, not cheaper-but-wrong)")
print("live cost_usd  bloated:", RESULTS["bloated_prompt"]["cost_usd"],
      "| skill:", RESULTS["skill"]["cost_usd"])

### Visual 1 — token cost: bloated vs skill

The bloated prompt is one giant block, always paid. The skill loads a tiny **router** up front and pulls the **one** knowledge file the matched task needs — not the full procedure or template. Watch: the skill's stack (router + knowledge) is well under the bloated block, and the printout above confirms the skill reached the *same* coded answer as the bloated prompt — cheaper *and* right, not cheaper-but-wrong. (Whether that answer's arithmetic is exact is the push-down lesson below.)

**Measure the token segments** for each approach from the real files — none hand-set. The bloated prompt is one inlined block; the skill splits into a router (always loaded) plus the one knowledge file it pulled when matched.

In [ ]:
# All counts measured from the real files / live run — none hand-set.
router_tok    = ntok(SKILL_ROUTER)     # front-matter router, loaded up front
knowledge_tok = ntok(SCRAP_CODES_MD)   # the one codes file pulled for this matched task

# stacked() wants {group: {segment: value}}; groups may carry different segments.
BUDGET = {
    "bloated_prompt": {"everything inlined (always)": RESULTS["bloated_prompt"]["loaded_tokens"]},
    "skill": {"router (always)": router_tok, "knowledge (when matched)": knowledge_tok},
}
bloated_total = sum(BUDGET["bloated_prompt"].values())
skill_total   = sum(BUDGET["skill"].values())
print(f"bloated {bloated_total} tok  ·  skill {skill_total} tok  ·  "
      f"bloated is {bloated_total / max(1, skill_total):.1f}x the skill")

In [ ]:
stacked(BUDGET, title="Tokens loaded per approach: bloated vs skill", ylabel="tokens loaded")

## Now walk a real SKILL.md

Tokens aside — what *is* the folder? The next cell prints the `scrap-coder/` tree and its `SKILL.md`. Watch for the 8 sections and the dual-clause `description` at the top — that one line is the router.

**A tiny tree printer** so we can see the skill's folder shape before reading its `SKILL.md`.

In [ ]:
def show_tree(p: Path, prefix: str = "") -> None:
    items = sorted(p.iterdir())
    for i, item in enumerate(items):
        connector = "└── " if i == len(items) - 1 else "├── "
        print(f"{prefix}{connector}{item.name}{'/' if item.is_dir() else ''}")
        if item.is_dir():
            show_tree(item, prefix + ("    " if i == len(items) - 1 else "│   "))

**Print the folder tree and the full `SKILL.md`.** Watch for the 8 sections and the dual-clause `description` at the top — that one line is the router.

In [ ]:
print(f"{SKILL_DIR.name}/")
show_tree(SKILL_DIR)
print("\n" + "=" * 70)
print(SKILL_MD)
print("=" * 70)

**The contract behind those 8 sections.** Each section earns its place; the highest-leverage line is the dual-clause `description` — the negative clause is what stops the router from firing the skill on the wrong task.

In [ ]:
print(dedent("""
    The 8 sections (the skill-creator contract):
      Files          — every file, what it holds, WHEN to load it
      Contract       — numbered, testable criteria (each maps to a test)
      Knowledge      — pointer to knowledge/ (loaded on demand)
      Steps          — one discrete action per step
      Output         — the concrete deliverable shape
      Antipatterns   — harvested from real failures, not imagined
      Validation     — how to prove it still works
      Examples       — smallest worked case

    The highest-leverage line is the YAML 'description': it carries BOTH a
    'Use when ...' clause AND a 'Do NOT use for ...' clause. That negative
    clause is what stops the router from firing the skill on the wrong task.
"""))

## Progressive disclosure, demonstrated

The next cell instruments the loader to log every file it reads, then runs two tasks: one that matches the skill and one that doesn't. Watch the matching task pull two files while the non-matching task loads nothing past the router line.

**Instrument the loader** so every file read is logged. `route()` checks whether the skill's description matches the task — the cheap "title" check before anything else loads.

In [ ]:
# Instrument the skill loader: log every file it reads.
LOADED = []

def skill_read(rel: str) -> str:
    """Read a file from the skill and record that we loaded it."""
    p = SKILL_DIR / rel
    LOADED.append(rel)
    return p.read_text() if p.exists() else f"(missing: {rel})"

def route(task: str) -> bool:
    """Does the skill's description match this task? (the 'title' check)."""
    desc = re.search(r"description:\s*\|?\s*\n((?:\s+.*\n)+)", SKILL_MD)
    desc_text = (desc.group(1) if desc else "").lower()
    triggers = ["scrap", "reject", "categorize", "code this"]
    return any(t in task.lower() and t in (desc_text + " scrap reject categorize") for t in triggers)

**Simulate disclosure for one task.** The router (`SKILL.md` front-matter) always loads — it's tiny and it's how the agent decides relevance. If the task matches, pull the one knowledge file the steps need; if not, the knowledge stays on disk.

In [ ]:
def disclose(task: str) -> list:
    """Progressive disclosure for one task; return the files loaded. The router
    (SKILL.md) is ALWAYS read; the knowledge file is pulled ONLY when the task matches."""
    LOADED.clear()
    skill_read("SKILL.md")                          # router — always loaded (cheap)
    if route(task):                                 # matched? pull the knowledge it needs
        skill_read("knowledge/scrap_codes.md")
    return list(LOADED)

**Run one matching and one non-matching task.** Both load the router; only the matching task pulls the knowledge file.

In [ ]:
matching     = disclose("Categorize the shift's scrap rejects and total the cost.")
nonmatching  = disclose("Summarize the storage incident on /scratch2 for the RCA.")

print("MATCHING task  -> loaded:", matching)
print("NON-MATCH task -> loaded:", nonmatching, "(router only — knowledge stayed on disk)")

### Visual 2 — files loaded per task

Show the title first; open the manual only if the job calls for it.

**Count files loaded per task.** Two bars; the matching task is the one we want the room watching, so it's the highlight.

In [ ]:
counts = {"matching task": len(matching), "non-matching task": len(nonmatching)}
print("files loaded:", counts)

In [ ]:
bars(counts, title="Progressive disclosure: files pulled per task",
     ylabel="files loaded into context", highlight="matching task")

## Push down to determinism

The cost total is the same math every time — a script's job, not the LLM's. The next cell runs the math two ways across the noisy rows (LLM in its head vs `total_cost.py`) and scores each against ground truth. Watch the LLM slip on a few messy rows while the script is exact on all of them.

**Load the deterministic core** — the script the skill pushes its math down to — and use it as the answer key. `truth_cost` gives ground truth for any single row.

In [ ]:
# Ground truth from the deterministic core (this is the answer key).
spec = importlib.util.spec_from_file_location("total_cost", SKILL_DIR / "scripts" / "total_cost.py")
total_cost = importlib.util.module_from_spec(spec)
spec.loader.exec_module(total_cost)
TRUTH = total_cost.categorize(NOISY_INPUTS)["total_cost"]

def truth_cost(row: str) -> float:
    """Ground-truth cost of a single row, from the deterministic core."""
    return total_cost.categorize([row])["total_cost"]

**A cent-exact match check** — 1 if an answer equals the expected figure to the cent, else 0.

In [ ]:
def matches(answer, expected) -> int:
    """1 if the answer matches expected (to the cent), else 0."""
    try:
        return int(round(float(answer), 2) == round(float(expected), 2))
    except (TypeError, ValueError):
        return 0

**Ask the model to do the math in its head** — one row, no calculator tool — so we can compare it against the script.

In [ ]:
async def llm_total(row: str):
    """Ask the model to total ONE row's scrap cost in its head (no calculator tool)."""
    result = await run(
        model,
        StaticProvider([NOOP_TOOL]),
        BLOATED_SYSTEM,
        f"Total the scrap cost for this row. Reply with ONLY the dollar number:\n{row}",
        max_turns=2,
    )
    m = re.search(r"-?\d+(?:\.\d+)?", (result.content or "").replace(",", ""))
    return float(m.group()) if m else None

**Score both modes against ground truth, row by row.** The script is exact on every row by construction; the LLM is checked row-for-row against the answer key.

In [ ]:
REL = {}
REL["script_does_math"] = [1] * len(NOISY_INPUTS)   # the script is exact on every row
REL["llm_does_math"]    = [matches(await llm_total(row), truth_cost(row)) for row in NOISY_INPUTS]

print("ground-truth total $", TRUTH)
print("LLM-math correct: %d/%d" % (sum(REL["llm_does_math"]), len(REL["llm_does_math"])))
print("script   correct: %d/%d" % (sum(REL["script_does_math"]), len(REL["script_does_math"])))

**The honest caveat, stated aloud.** How often the LLM slips is *model-dependent* — the durable claim is that moving a deterministic step into a tool is the harness win, not "LLMs can't add".

In [ ]:
print("Honest caveat: how often the LLM slips is MODEL-DEPENDENT. A frontier")
print("model on clean rows may get every one right. The durable claim isn't")
print("'LLMs can't add' — it's that moving a deterministic step into a tool is")
print("the harness win (scaffolding swings 20+ pts on identical SWE-bench weights).")

### Visual 3 — reliability & cost: LLM step vs script step

Two charts: accuracy across the noisy set, then tokens per run. The script wins both. *If a script can do it, a script does it.*

**Compute accuracy and cost** for each mode, from the live scores.

In [ ]:
n = len(NOISY_INPUTS)
acc = {m: 100.0 * sum(REL[m]) / n for m in ["llm_does_math", "script_does_math"]}
tpr = {"llm_does_math": ntok("\n".join(NOISY_INPUTS)) + 40, "script_does_math": 0}

ACCURACY = {"LLM does math": acc["llm_does_math"], "script does math": acc["script_does_math"]}
COST     = {"LLM does math": tpr["llm_does_math"], "script does math": tpr["script_does_math"]}
print("accuracy %:", ACCURACY, "| tokens/run:", COST)

**Accuracy on the noisy set** — the script is exact on every row; the LLM slips on the messy ones.

In [ ]:
compare(ACCURACY, title="Accuracy on the noisy set: LLM vs script",
        ylabel="% rows totaled correctly", fmt="{:.0f}%")

**Cost per run** — the pushed-down script spends zero model tokens on the math step.

In [ ]:
compare(COST, title="Cost per run: LLM vs script", ylabel="tokens per run (runtime)")

## The contract is the promise → tests keep it

Each numbered contract line maps to a test. The next cell runs the validator and the unit tests live, then **breaks the skill on purpose** — stripping the `Do NOT use for` clause — and re-runs the validator. Watch it pass, then FAIL with the exact reason, then pass again after the restore.

**The validator** — mechanical gates: name matches the folder, the 8 sections appear in order, the description has both clauses, required subdirs exist, and every script has a test.

In [ ]:
def validate_skill(skill_dir: Path) -> dict:
    """Mechanical gates: name match, 8 sections in order, dual-clause description,
    required subdirs, checklist, every script has a test."""
    errs = []
    sk = skill_dir / "SKILL.md"
    if not sk.exists():
        return {"ok": False, "errors": ["SKILL.md missing"]}
    text = sk.read_text()

    name = re.search(r"^name:\s*(\S+)", text, re.M)
    if name and name.group(1) != skill_dir.name:
        errs.append(f'name "{name.group(1)}" != folder "{skill_dir.name}"')

    expected = ["Files", "Contract", "Knowledge", "Steps", "Output",
                "Antipatterns", "Validation", "Examples"]
    found = [m for m in re.findall(r"^## (\w+)", text, re.M) if m in expected]
    if found != expected:
        errs.append(f"sections wrong: got {found}")

    desc = re.search(r"description:\s*\|?\s*\n((?:\s+.*\n)+)", text)
    dtext = (desc.group(1) if desc else "").lower()
    if "use when" not in dtext:
        errs.append('description missing positive clause ("Use when ...")')
    if "do not use" not in dtext.replace("not use for", "do not use"):
        errs.append('description missing negative clause ("Do NOT use for ...")')

    for d in ["scripts", "templates", "knowledge", "validation", "tests", "examples"]:
        if not (skill_dir / d).is_dir():
            errs.append(f"missing required subdir: {d}/")
    if not (skill_dir / "validation" / "quality-checklist.md").exists():
        errs.append("validation/quality-checklist.md missing")
    for script in (skill_dir / "scripts").glob("*.py"):
        if not (skill_dir / "tests" / "unit" / f"test_{script.stem}.py").exists():
            errs.append(f"no test for scripts/{script.name}")
    return {"ok": not errs, "errors": errs}

def report(verdict):
    if verdict["ok"]:
        print("validate_skill: PASS — all gates green.")
    else:
        print("validate_skill: FAIL")
        for e in verdict["errors"]:
            print("  •", e)

**Run the validator** on the real skill — expect all gates green.

In [ ]:
report(validate_skill(SKILL_DIR))

**Run the unit tests live** via pytest, in an isolated subprocess so kernel state can't leak in.

In [ ]:
# Use the pytest on PATH if present, else this interpreter's "-m pytest".
import shutil
_pytest = shutil.which("pytest")
_cmd = [_pytest] if _pytest else [sys.executable, "-m", "pytest"]
res = subprocess.run(_cmd + [str(SKILL_DIR / "tests" / "unit"), "-q"],
                     capture_output=True, text=True)
print("pytest tests/unit  ·  exit =", res.returncode)
print((res.stdout + res.stderr).strip()[-600:])

**Now break it on purpose.** Strip the `Do NOT use for` clause and re-validate — watch it FAIL loudly with the exact reason, then PASS again after we restore the file.

In [ ]:
print("--- breaking the skill: remove the 'Do NOT use for' clause ---")
backup = (SKILL_DIR / "SKILL.md").read_text()
broken = re.sub(r"Do NOT use for[^\n]*", "", backup)
(SKILL_DIR / "SKILL.md").write_text(broken)
report(validate_skill(SKILL_DIR))     # -> FAIL, loudly, with the reason

# Restore.
(SKILL_DIR / "SKILL.md").write_text(backup)
print("\nrestored. re-validate:")
report(validate_skill(SKILL_DIR))

### Visual 4 — the four test layers

The skill-creator's four layers, each catching a different bug class, shown green when they ran (or are expected to). *An untested contract is no contract.*

**Assemble the test-layer status.** The unit layer reflects this run's live pytest result; the other three come from the precomputed map (they need the full harness).

In [ ]:
# The test pyramid a skill should have. This demo ships UNIT tests (run live above);
# the higher layers are the contract you'd add — shown by what's actually present, not faked.
order   = ["unit", "integration", "evals", "e2e"]
catches = {"unit": "logic bugs", "integration": "wiring bugs",
           "evals": "quality regressions", "e2e": "workflow breaks"}
present = {l: (SKILL_DIR / "tests" / l).is_dir() for l in order}
status  = {"unit": present["unit"] and res.returncode == 0,
           "integration": present["integration"],
           "evals": present["evals"],
           "e2e": present["e2e"]}
row_labels = [f"{l}  ({catches[l]})" for l in order]
grid       = [[status[l]] for l in order]
print("layer status:", status)

In [ ]:
matrix(row_labels, ["passed?"], grid, title="Four test layers — each catches a different bug class")

## Iterative creation with skill-creator

You don't write a skill cold. The next cell scaffolds a fresh stub (`coa-intake`), validates it (FAIL — incomplete), fills the description and adds one test, then re-validates (PASS). Watch the validator drive the 5-step loop from red to green.

**Scaffold a fresh stub** — a new `coa-intake` skill with all 8 section headers but a placeholder description and a script with no test. Both gaps are deliberate: the validator must FAIL.

In [ ]:
import shutil as _sh

# Scaffold a brand-new tiny skill: coa-intake (parse a Certificate of Analysis).
# Start from a clean slate so the demo is deterministic on re-runs.
NEW = SKILLS_DIR / "coa-intake"
if NEW.exists():
    _sh.rmtree(NEW)
for d in ["scripts", "templates", "knowledge", "validation", "tests/unit", "examples/simple"]:
    (NEW / d).mkdir(parents=True, exist_ok=True)

# Stub SKILL.md: 8 headers, placeholder description (no real clauses), script with NO test.
(NEW / "SKILL.md").write_text(dedent("""\
    ---
    name: coa-intake
    version: 0.1.0
    description: |
      Placeholder. Fill this in before the skill is usable.
    ---

    # coa-intake

    ## Files
    ## Contract
    ## Knowledge
    ## Steps
    ## Output
    ## Antipatterns
    ## Validation
    ## Examples
    """))
(NEW / "validation" / "quality-checklist.md").write_text("# checklist\n- [ ] TODO\n")
(NEW / "scripts" / "parse_coa.py").write_text("def parse_coa(text):\n    return {}\n")

print("scaffold:")
show_tree(NEW)

**Validate the stub — expect FAIL.** A placeholder description and a script with no test should both trip the gates.

In [ ]:
print("1st validate (expect FAIL — placeholder description, no test for the script):")
v1 = validate_skill(NEW)
report(v1)
assert not v1["ok"], "scaffold demo broken: stub should FAIL validation"

**Fix the two gaps** — fill the dual-clause description and add the missing unit test — the way you actually build a skill, red to green.

In [ ]:
# Fill the description (dual clause) + add the missing test.
md_text = (NEW / "SKILL.md").read_text().replace(
    "Placeholder. Fill this in before the skill is usable.",
    'Parse a supplier Certificate of Analysis (CoA) into structured lot/spec '
    'fields. Use when asked to "read this CoA" or "extract the CoA values". '
    'Do NOT use for invoices or packing slips — use doc-intake for those.')
(NEW / "SKILL.md").write_text(md_text)
(NEW / "tests" / "unit" / "test_parse_coa.py").write_text(dedent("""\
    import sys
    from pathlib import Path
    sys.path.insert(0, str(Path(__file__).resolve().parents[2] / "scripts"))
    from parse_coa import parse_coa

    def test_returns_dict():
        assert isinstance(parse_coa("lot: 42"), dict)
    """))

**Re-validate — expect PASS.** With the description filled and the test added, every gate goes green.

In [ ]:
print("2nd validate (after filling description + adding a test):")
v2 = validate_skill(NEW)
report(v2)
assert v2["ok"], "scaffold demo broken: filled skill should PASS validation"

## Your turn

In [13]:
# ✅ TODO — turn YOUR bloated prompt into a skill.
#
# 1. Paste a system prompt you actually use into MY_PROMPT below.
# 2. Scaffold a folder: pull the DETERMINISTIC bits into scripts/, the
#    REFERENCE bits into knowledge/, leave only JUDGMENT in Steps.
# 3. Write the dual-clause description (Use when ... / Do NOT use for ...).
# 4. Run validate_skill() until it passes.
#
MY_PROMPT = """
PASTE YOUR SYSTEM PROMPT HERE
"""

MY_SKILL = SKILLS_DIR / "my-skill"
for d in ["scripts", "templates", "knowledge", "validation", "tests/unit", "examples/simple"]:
    (MY_SKILL / d).mkdir(parents=True, exist_ok=True)

# (MY_SKILL / "SKILL.md").write_text(...)   # 8 sections; description first.
# report(validate_skill(MY_SKILL))

print("scaffold ready at:", MY_SKILL)
print("Fill SKILL.md (8 sections), add a test, then run: report(validate_skill(MY_SKILL))")


scaffold ready at: skills/my-skill
Fill SKILL.md (8 sections), add a test, then run: report(validate_skill(MY_SKILL))


## Takeaway

> A skill is **procedural memory engineered into a folder**: progressive
> disclosure keeps it cheap, push-down keeps it reliable, the contract + tests
> keep it honest, and semver + git make it shippable across the lab.

Next: the context that shapes *judgment* rather than procedure — the identity file.
